# Analyse des municipales 2014-2020-2026 et preparation de la prediction 2032

## Problematique
Predire le nombre de votants et le taux de participation aux elections municipales de 2032 a partir des resultats 2014, 2020 et 2026, en tenant compte du caractere atypique de 2020 dans le contexte Covid-19.

## Objectifs du notebook
- charger et inspecter les jeux de donnees communaux ;
- harmoniser les colonnes utiles aux jointures et aux calculs electoraux ;
- construire un panel final par commune et par annee ;
- produire des visualisations propres et lisibles ;
- montrer que 2020 ne doit pas etre interprete comme une simple tendance lineaire ;
- preparer un dataset machine learning et comparer quelques approches simples pour 2032.

## Arborescence attendue
```text
Data prediction/
|-- analysis_municipales_2032.ipynb
|-- README.md
|-- dataset_final.csv
|-- data/
|   |-- elections_2014.csv
|   |-- elections_2020.csv
|   |-- elections_2026.csv
|   |-- population.csv
|   |-- chomage.csv
|   |-- revenus_pauvrete.csv
|   '-- crimes_delits.csv
|-- outputs/
|   '-- figures/
'-- src/
    |-- load_data.py
    |-- preprocess.py
    |-- visualize.py
    '-- model.py
```

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from src.load_data import (
    build_dataset_catalog,
    build_inspection_table,
    load_expected_datasets,
    missing_values_table,
)
from src.preprocess import (
    add_period_label,
    analyze_2020_atypicality,
    build_election_panel,
    build_final_dataset,
    compute_commune_variation,
    compute_descriptive_statistics,
    compute_yearly_summary,
    preprocess_contextual_dataframe,
    summarize_final_dataset,
)
from src.visualize import (
    plot_2020_anomaly,
    plot_boxplot_by_year,
    plot_correlation_heatmap,
    plot_histogram_by_year,
    plot_scatter_relationship,
    plot_top_communes_bar,
    plot_yearly_mean_line,
)
from src.model import (
    compare_model_scenarios,
    generate_projection_2032,
    infer_feature_columns,
    prepare_ml_dataset,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 120)

project_root = Path.cwd()
data_dir = project_root / "data"
figures_dir = project_root / "outputs" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)
dataset_output_path = project_root / "dataset_final.csv"
projection_output_path = project_root / "outputs" / "projection_2032_exploratoire.csv"

print(f"Racine du projet : {project_root}")
print(f"Dossier des donnees : {data_dir}")
print(f"Dossier des figures : {figures_dir}")

## 1. Chargement et inspection des donnees

Cette premiere etape verifie la presence des fichiers, inspecte leur structure, repere les colonnes candidates aux jointures et quantifie les valeurs manquantes. L'objectif est de comprendre rapidement l'etat brut des donnees avant tout nettoyage.

In [ ]:
catalog = build_dataset_catalog(data_dir)
print("Catalogue des fichiers attendus :")
print(catalog.to_string(index=False))

datasets, missing_files = load_expected_datasets(data_dir)

if missing_files:
    print("\nFichiers manquants detectes :")
    for dataset_name, file_path in missing_files.items():
        print(f"- {dataset_name}: {file_path}")

inspection_table = build_inspection_table(datasets)
print("\nTable d'inspection synthetique :")
print(inspection_table.to_string(index=False))

for dataset_name, df in datasets.items():
    print("\n" + "=" * 100)
    print(f"Dataset : {dataset_name}")
    print(f"Dimensions : {df.shape[0]} lignes x {df.shape[1]} colonnes")
    print("Apercu des 3 premieres lignes :")
    print(df.head(3).to_string(index=False))
    print("\nTypes de colonnes (15 premieres) :")
    print(df.dtypes.astype(str).head(15).to_string())
    print("\nValeurs manquantes les plus importantes :")
    print(missing_values_table(df).head(10).to_string(index=False))

## 2. Nettoyage et preparation

Le nettoyage vise ici trois objectifs :

- harmoniser les identifiants communaux ;
- calculer les variables electorales utiles a l'analyse ;
- preparer les jeux contextuels pour une fusion stable, meme si les noms de colonnes varient d'une source a l'autre.

Le code utilise une detection souple des colonnes afin d'eviter de crasher quand un fichier n'emploie pas exactement les memes intitules que les autres.

In [ ]:
required_election_files = {
    "elections_2014": 2014,
    "elections_2020": 2020,
    "elections_2026": 2026,
}

missing_elections = [name for name in required_election_files if name not in datasets]
if missing_elections:
    raise FileNotFoundError(
        "Les fichiers electoraux suivants sont indispensables : "
        + ", ".join(missing_elections)
    )

election_inputs = {
    year: datasets[file_name]
    for file_name, year in required_election_files.items()
}

election_panel, election_metadata = build_election_panel(election_inputs)
election_panel = add_period_label(election_panel)

print("Panel electoral preprocesse :")
print(election_panel.head().to_string(index=False))

metadata_elections_df = (
    pd.DataFrame.from_dict(election_metadata, orient="index")
    .reset_index()
    .rename(columns={"index": "annee"})
)
print("\nMetadonnees de preparation des elections :")
print(metadata_elections_df.to_string(index=False))

contextual_processed = {}
contextual_metadata = []
for dataset_name in ["population", "chomage", "revenus_pauvrete", "crimes_delits"]:
    if dataset_name in datasets:
        processed_df, metadata = preprocess_contextual_dataframe(datasets[dataset_name], dataset_name)
        contextual_processed[dataset_name] = processed_df
        contextual_metadata.append(metadata)

if contextual_metadata:
    contextual_metadata_df = pd.DataFrame(contextual_metadata)
    print("\nMetadonnees de preparation des datasets contextuels :")
    print(contextual_metadata_df.to_string(index=False))
else:
    print("\nAucun dataset contextuel n'a ete charge.")

## 3. Fusion des datasets

La fusion se fait par commune, et par annee quand l'information existe dans la source contextuelle. Le rapport de fusion permet d'identifier les pertes de lignes ou les jointures non resolues. Cette etape est importante pour juger de la qualite du panel final utilise ensuite pour l'analyse et la prediction.

In [ ]:
dataset_final, merge_report = build_final_dataset(election_panel, contextual_processed)
summary_final = summarize_final_dataset(dataset_final)

dataset_final.to_csv(dataset_output_path, index=False, encoding="utf-8-sig")

print("Rapport de fusion :")
print(merge_report.to_string(index=False))

print("\nResume du dataset final :")
print(summary_final.to_string(index=False))

print("\nColonnes du dataset final :")
print(dataset_final.columns.tolist())

print("\nApercu du dataset final :")
print(dataset_final.head().to_string(index=False))

print(f"\nLe dataset final a ete exporte vers : {dataset_output_path}")

## 4. Analyse exploratoire

Cette section produit un premier diagnostic statistique global puis compare explicitement 2014, 2020 et 2026. L'accent est mis sur le nombre de votants et le taux de participation, car ce sont les deux cibles du futur travail predictif.

In [ ]:
global_stats = compute_descriptive_statistics(dataset_final)
yearly_summary = compute_yearly_summary(dataset_final)

print("Statistiques descriptives globales des variables numeriques :")
if global_stats.empty:
    print("Aucune statistique globale disponible.")
else:
    print(global_stats.to_string())

print("\nComparaison 2014 vs 2020 vs 2026 :")
if yearly_summary.empty:
    print("Resume annuel indisponible.")
else:
    print(yearly_summary.to_string())

baisse_2014_2020 = compute_commune_variation(
    dataset_final, metric="taux_participation", year_from=2014, year_to=2020
)
reprise_2020_2026 = compute_commune_variation(
    dataset_final, metric="taux_participation", year_from=2020, year_to=2026
)

if not baisse_2014_2020.empty:
    print("\nTop 10 des communes avec la plus forte baisse de participation entre 2014 et 2020 :")
    print(baisse_2014_2020.head(10).to_string(index=False))

if not reprise_2020_2026.empty:
    print("\nTop 10 des communes avec la plus forte reprise de participation entre 2020 et 2026 :")
    print(
        reprise_2020_2026.sort_values("variation_absolue", ascending=False)
        .head(10)
        .to_string(index=False)
    )

if {"annee", "taux_participation"}.issubset(dataset_final.columns):
    yearly_means = dataset_final.groupby("annee")["taux_participation"].mean()
    if {2014, 2020, 2026}.issubset(set(yearly_means.index)):
        diff_20_14 = yearly_means.loc[2020] - yearly_means.loc[2014]
        diff_26_20 = yearly_means.loc[2026] - yearly_means.loc[2020]
        print("\nInterpretation automatique :")
        print(f"- L'ecart moyen de participation entre 2014 et 2020 est de {diff_20_14:.2f} points.")
        print(f"- L'ecart moyen de participation entre 2020 et 2026 est de {diff_26_20:.2f} points.")
        if diff_20_14 < 0 and diff_26_20 > 0:
            print(
                "- La sequence baisse en 2020 puis reprise en 2026 suggere une rupture ponctuelle plus qu'une tendance lineaire reguliere."
            )

## 5. Visualisations demandees

Les graphiques sont enregistres automatiquement dans `outputs/figures/`. La palette choisie reste sobre : bleu pour 2014, rouge attenue pour l'annee atypique 2020, vert pour 2026. Cela permet de faire ressortir visuellement l'anomalie sans tomber dans une palette trop vive.

In [ ]:
if "taux_participation" in dataset_final.columns:
    plot_histogram_by_year(
        dataset_final,
        column="taux_participation",
        output_path=figures_dir / "histogramme_taux_participation_par_annee.png",
    )
    plot_boxplot_by_year(
        dataset_final,
        column="taux_participation",
        output_path=figures_dir / "boxplot_taux_participation_par_annee.png",
    )
    plot_yearly_mean_line(
        dataset_final,
        metric="taux_participation",
        output_path=figures_dir / "courbe_moyenne_taux_participation.png",
    )

if "nombre_votants" in dataset_final.columns:
    plot_yearly_mean_line(
        dataset_final,
        metric="nombre_votants",
        output_path=figures_dir / "courbe_moyenne_nombre_votants.png",
    )

if {"population_totale", "taux_participation"}.issubset(dataset_final.columns):
    plot_scatter_relationship(
        dataset_final,
        x_column="population_totale",
        y_column="taux_participation",
        output_path=figures_dir / "scatter_population_vs_participation.png",
    )

if {"taux_chomage", "taux_participation"}.issubset(dataset_final.columns):
    plot_scatter_relationship(
        dataset_final,
        x_column="taux_chomage",
        y_column="taux_participation",
        output_path=figures_dir / "scatter_chomage_vs_participation.png",
    )

if {"revenu_median", "taux_participation"}.issubset(dataset_final.columns):
    plot_scatter_relationship(
        dataset_final,
        x_column="revenu_median",
        y_column="taux_participation",
        output_path=figures_dir / "scatter_revenus_vs_participation.png",
    )

heatmap_columns = [
    column
    for column in [
        "nombre_inscrits",
        "nombre_votants",
        "taux_participation",
        "population_totale",
        "taux_chomage",
        "revenu_median",
        "taux_pauvrete",
        "nombre_faits",
        "taux_criminalite",
    ]
    if column in dataset_final.columns
]
plot_correlation_heatmap(
    dataset_final,
    columns=heatmap_columns,
    output_path=figures_dir / "heatmap_correlation_variables_numeriques.png",
)

if not baisse_2014_2020.empty:
    plot_top_communes_bar(
        baisse_2014_2020,
        output_path=figures_dir / "bar_baisse_2014_2020.png",
        title="Top 10 des communes avec la plus forte baisse entre 2014 et 2020",
        n=10,
        ascending=True,
    )

if not reprise_2020_2026.empty:
    plot_top_communes_bar(
        reprise_2020_2026,
        output_path=figures_dir / "bar_reprise_2020_2026.png",
        title="Top 10 des communes avec la plus forte reprise entre 2020 et 2026",
        n=10,
        ascending=False,
    )

print(f"Les figures ont ete enregistrees dans : {figures_dir}")

## 6. Analyse specifique de l'effet atypique 2020

Cette section isole volontairement l'annee 2020. Le but n'est pas seulement de constater un niveau de participation plus bas, mais de montrer pourquoi cette annee ne doit pas etre lue comme une simple continuation de la trajectoire 2014-2026.

In [ ]:
atypicality = analyze_2020_atypicality(dataset_final)
summary_2020 = atypicality["summary"]
relative_gaps_2020 = atypicality["relative_gaps"]
tests_2020 = atypicality["tests"]

print("Resume moyenne / mediane du taux de participation par annee :")
print(summary_2020.to_string(index=False) if not summary_2020.empty else "Aucun resume disponible.")

print("\nEcarts relatifs de 2020 par rapport aux autres annees :")
print(relative_gaps_2020.to_string(index=False) if not relative_gaps_2020.empty else "Aucun ecart calcule.")

print("\nTests simples de rupture sur les communes apparies :")
print(tests_2020.to_string(index=False) if not tests_2020.empty else "Pas assez d'observations apparies pour realiser des tests.")

if not summary_2020.empty:
    plot_2020_anomaly(
        summary_2020,
        output_path=figures_dir / "anomalie_2020_participation.png",
    )

if not relative_gaps_2020.empty:
    print("\nLecture rapide des ecarts :")
    for _, row in relative_gaps_2020.iterrows():
        print(
            f"- {row['comparaison']} : {row['ecart_moyenne_points']:.2f} points, soit {row['ecart_relatif_pct']:.2f}% en relatif."
        )

if not tests_2020.empty:
    significant_rows = tests_2020[
        (tests_2020["t_test_pvalue"] < 0.05)
        | (tests_2020["wilcoxon_pvalue"] < 0.05)
    ]
    if not significant_rows.empty:
        print("\nInterpretation statistique : les tests suggerent une difference significative entre 2020 et les annees de comparaison pour les communes appariees.")
    else:
        print("\nInterpretation statistique : la rupture n'est pas fortement confirmee par les tests retenus, mais la lecture descriptive peut rester tres informative.")

print("\nConclusion analytique : 2020 doit etre traite avec prudence. Dans un cadre predictif, il est preferable soit d'introduire une variable explicite `covid_2020`, soit de reduire son poids, soit de comparer un scenario avec et sans 2020.")

## 7. Preparation a la prediction 2032

Nous preparons ici un dataset machine learning simple. Les modeles proposes ne pretendent pas fournir une prevision definitive ; ils servent surtout a poser une base methodologique solide pour la suite du projet. Trois strategies sont comparees :

- `avec_covid` : 2020 est conserve comme une annee normale mais avec une variable binaire explicite ;
- `covid_pondere` : 2020 est conserve mais avec un poids plus faible a l'entrainement ;
- `sans_2020` : 2020 est retire du jeu d'apprentissage pour tester un scenario sans annee atypique.

Remarque importante : si `nombre_inscrits` n'est pas connu a l'avance pour 2032, il faudra soit le projeter en amont, soit l'exclure de la modelisation finale.

In [ ]:
ml_dataset = prepare_ml_dataset(dataset_final)

print("Apercu du dataset machine learning :")
print(ml_dataset.head().to_string(index=False))

for target in ["nombre_votants", "taux_participation"]:
    selected_features = infer_feature_columns(ml_dataset, target)
    print(f"\nVariables explicatives retenues pour {target} :")
    print(selected_features if selected_features else "Aucune variable disponible.")

In [ ]:
results_votants, bundles_votants = compare_model_scenarios(
    ml_dataset,
    target_column="nombre_votants",
    covid_weight=0.5,
)
results_participation, bundles_participation = compare_model_scenarios(
    ml_dataset,
    target_column="taux_participation",
    covid_weight=0.5,
)

print("Comparaison des approches pour la cible `nombre_votants` :")
if results_votants.empty:
    print("Pas assez de donnees pour entrainer les modeles sur `nombre_votants`.")
else:
    print(results_votants.to_string(index=False))

print("\nComparaison des approches pour la cible `taux_participation` :")
if results_participation.empty:
    print("Pas assez de donnees pour entrainer les modeles sur `taux_participation`.")
else:
    print(results_participation.to_string(index=False))

def comment_best_result(results_df, target_name):
    if results_df.empty:
        return
    best_row = results_df.iloc[0]
    print(
        f"Meilleure combinaison provisoire pour {target_name} : {best_row['modele']} | scenario={best_row['scenario']} | RMSE={best_row['rmse']:.3f} | MAE={best_row['mae']:.3f} | R2={best_row['r2']:.3f}"
    )

print("\nLecture rapide des modeles :")
comment_best_result(results_votants, "nombre_votants")
comment_best_result(results_participation, "taux_participation")

In [ ]:
projection_votants = generate_projection_2032(
    ml_dataset,
    bundles_votants,
    target_column="nombre_votants",
    projection_year=2032,
    preferred_scenario="covid_pondere",
)
projection_participation = generate_projection_2032(
    ml_dataset,
    bundles_participation,
    target_column="taux_participation",
    projection_year=2032,
    preferred_scenario="covid_pondere",
)

if not projection_votants.empty:
    projection_votants = projection_votants.rename(
        columns={
            "scenario": "scenario_nombre_votants",
            "modele": "modele_nombre_votants",
        }
    )

if not projection_participation.empty:
    projection_participation = projection_participation.rename(
        columns={
            "scenario": "scenario_taux_participation",
            "modele": "modele_taux_participation",
        }
    )

if not projection_votants.empty and not projection_participation.empty:
    merge_keys = [
        column
        for column in ["code_commune", "nom_commune", "code_departement", "annee"]
        if column in projection_votants.columns and column in projection_participation.columns
    ]
    projection_2032 = projection_votants.merge(
        projection_participation,
        how="outer",
        on=merge_keys,
    )
elif not projection_votants.empty:
    projection_2032 = projection_votants.copy()
elif not projection_participation.empty:
    projection_2032 = projection_participation.copy()
else:
    projection_2032 = pd.DataFrame()

if projection_2032.empty:
    print("La projection exploratoire 2032 n'a pas pu etre generee.")
else:
    projection_2032.to_csv(projection_output_path, index=False, encoding="utf-8-sig")
    print("Projection exploratoire 2032 :")
    print(projection_2032.head(20).to_string(index=False))
    print(f"\nLe fichier de projection a ete enregistre ici : {projection_output_path}")

    if "prediction_taux_participation_2032" in projection_2032.columns:
        print("\nSynthese exploratoire :")
        print(
            f"- Taux de participation moyen projete en 2032 : {projection_2032['prediction_taux_participation_2032'].mean():.2f}%"
        )
    if "prediction_nombre_votants_2032" in projection_2032.columns:
        print(
            f"- Nombre moyen de votants projete en 2032 : {projection_2032['prediction_nombre_votants_2032'].mean():.0f}"
        )

print("\nRappel methodologique : cette projection reste exploratoire. Elle prolonge le dernier contexte observe par commune et ne remplace pas une vraie projection des variables explicatives vers 2032.")

## 8. Interpretation finale et limites

### Ce que l'analyse doit permettre de montrer
- Les municipales 2014, 2020 et 2026 ne racontent pas la meme histoire en termes de participation.
- L'annee 2020 doit etre traitee comme une annee atypique plutot que comme une simple etape d'une tendance reguliere.
- Les variables contextuelles comme la population, le chomage, le revenu median, la pauvrete ou les faits de delinquance peuvent aider a expliquer une partie des variations de participation.

### Limites
- La qualite de l'analyse depend fortement de la qualite des CSV sources et de la precision des jointures communales.
- Avec seulement trois elections municipales, la modelisation reste preparatoire et exploratoire.
- Les projections 2032 deviennent plus solides si l'on dispose de scenarios futurs sur la population, les inscrits et les variables socio-economiques.

### Pistes d'amelioration
- enrichir le projet avec des variables territoriales, politiques ou geographiques ;
- construire des modeles par taille de commune ou par departement ;
- tester des validations temporelles plus strictes ;
- produire une note methodologique pour justifier le traitement specifique de 2020 devant le jury.